# Kubernetes cluster

## Prerequisites
* You have successfully followed the [Provisioning Resources tutorial](./provision.ipynb)
* You have basic Python knowledge
* You have basic kubernetes knowledge

## Introduction

The orchestrator relies on kubernetes to deploy most of its functions. An
essential step is then to create a kubernetes cluster composed of all the nodes
defined in the deployement model (more exactely all the ones flagged as
`is_provisioned`).

To do so, we can use the `K8SManagement` manager, which role is to deploy and
configure a kubernetes cluster compatible with the 5G network deployement that
we want.

In this tutorial, it will be composed of the `node1`, `node2`, and `node3`
machines provisioned that we provisioned in the
[Provisioning Resources tutorial](./provision.ipynb).

The nodes are label ia way that the workload is scheduled correctly on them
(e.g., UEs can only be deployed on `node-ues`). Internally, the library set the
label `node-type.5g.blueprint` to the nodes in the cluster. For example, if a
node can be a `ran`, it has the label `node-type.5g.blueprint/ran=true`.

As mentionned earlier, managers consume metamodel and deployment model. Here we
want to use the output of the provisioning we did in the previous step. When we
instantiate our kubernetes manager, we just have to mention which manager
preceeds it (in this case `prov`) and it will automatically retrieve its models.

However, as notebooks do not share their variables, we don't directly have
access to the `prov` instance of the *provisioning* notebook. However, as we
saved the metamodel and deployement model in the files
`state/provision/metamodel.prov.yaml` and
`state/provision/deployment_model.prov.yaml`, respectively, we can create
a new identical provisioner instance as follows.

In [1]:
from orchestrator.deployment import ProvisioningManagement

PROV_DIR="state/provision"
METAMODEL_FILE = f"{PROV_DIR}/metamodel.prov.yaml"
DEPLOYMENT_MODEL_FILE = f"{PROV_DIR}/deployment_model.prov.yaml"

prov = ProvisioningManagement(metamodel_file=METAMODEL_FILE, deployment_model_file=DEPLOYMENT_MODEL_FILE)



**IMPORTANT:** our orchestrator library follows a declarative and functional
approach, as such, the metamodel and deployment model are copied when passed
from one manager to another meaning that the input models are never modified by
the manager.

Once the manager is properly instantiated, we trigger the deployment of the
cluster on the machines. It will create a kubernetes cluster with all the nodes
and install the corresponding `kubeconfig` file on all the machines to simplify
later operations.

**NOTE:** to onboard existing kubernetes clusters please refer to the
[orchestrator](https://gitlab.inria.fr/slices-ri/blueprints/post-5g/orchestrator)
library documentation.

In [2]:
from orchestrator.deployment import K8SManagement

k8s = K8SManagement(metamodel_file=METAMODEL_FILE, deployment_model_file=DEPLOYMENT_MODEL_FILE)
await k8s.deploy(quiet=True)            # don't show output


Once it is deployed we can verify that the kubernets cluster is deployed and 
setup correctly by running the `kubectl` command.

In [6]:
print((await prov.run_node_command(
    command=["kubectl", "get", "nodes", "-o", "wide", "--show-labels"],
    nodes=["node1"],
    become=True, quiet=True))["node1"])

NAME    STATUS   ROLES           AGE     VERSION   INTERNAL-IP      EXTERNAL-IP   OS-IMAGE             KERNEL-VERSION             CONTAINER-RUNTIME    LABELS
node1   Ready    control-plane   7m45s   v1.36.2   10.123.230.195   <none>        Ubuntu 24.04.3 LTS   6.8.0-85-generic (amd64)   containerd://2.2.1   beta.kubernetes.io/arch=amd64,beta.kubernetes.io/os=linux,kubernetes.io/arch=amd64,kubernetes.io/hostname=node1,kubernetes.io/os=linux,node-role.kubernetes.io/control-plane=,node-type.5g.blueprint/core=true,node.kubernetes.io/exclude-from-external-load-balancers=
node2   Ready    <none>          7m      v1.36.2   10.123.59.108    <none>        Ubuntu 24.04.3 LTS   6.8.0-85-generic (amd64)   containerd://2.2.1   beta.kubernetes.io/arch=amd64,beta.kubernetes.io/os=linux,kubernetes.io/arch=amd64,kubernetes.io/hostname=node2,kubernetes.io/os=linux,node-type.5g.blueprint/ran=true
node3   Ready    <none>          7m      v1.36.2   10.123.105.195   <none>        Ubuntu 24.04.3 LTS   6.8.0-

In [5]:
print((await prov.run_node_command(
    command=["kubectl", "get", "all", "-A", "-o", "wide"],
    nodes=["node1"],
    become=True, quiet=True))["node1"])

NAMESPACE         NAME                                          READY   STATUS    RESTARTS        AGE     IP               NODE    NOMINATED NODE   READINESS GATES
calico-system     pod/calico-kube-controllers-dcb8f67cb-6msbc   1/1     Running   0               5m8s    10.244.104.1     node2   <none>           <none>
calico-system     pod/calico-node-ktcmr                         1/1     Running   0               4m59s   10.123.105.195   node3   <none>           <none>
calico-system     pod/calico-node-kz6gt                         1/1     Running   0               4m59s   10.123.59.108    node2   <none>           <none>
calico-system     pod/calico-node-wqdch                         1/1     Running   0               4m49s   10.123.230.195   node1   <none>           <none>
calico-system     pod/calico-typha-6947bbf55d-2p7km             1/1     Running   0               4m52s   10.123.59.108    node2   <none>           <none>
calico-system     pod/calico-typha-6947bbf55d-gdxqp          